In [1]:
import sys

print(sys.executable)

C:\Users\zixia\anaconda3\envs\cemct\python.exe


In [2]:
from pathlib import Path


file_path = Path(
    r"C:\Users\zixia\XseeO2_coding_folder\synthetic_labelled_cement_4phase_100.tif"
)

print("Input file:", file_path)
print("File exists:", file_path.exists())

Input file: C:\Users\zixia\XseeO2_coding_folder\synthetic_labelled_cement_4phase_100.tif
File exists: True


In [3]:
import cemct
import openpnm
import porespy
import pyamg

print("CemCT:", cemct.__file__)
print("OpenPNM:", openpnm.__version__)
print("PoreSpy:", porespy.__version__)
print("PyAMG:", pyamg.__version__)

CemCT: C:\Users\zixia\XseeO2_coding_folder\CemCT\src\cemct\__init__.py
OpenPNM: 3.6.2
PoreSpy: 3.0.4
PyAMG: 5.3.0


In [4]:
import numpy as np

from cemct.io import load_labelled_tiff


volume = load_labelled_tiff(file_path)

print("Volume shape (Z, Y, X):", volume.shape)
print("Data type:", volume.dtype)
print("Observed labels:", np.unique(volume))

Volume shape (Z, Y, X): (100, 100, 100)
Data type: uint8
Observed labels: [1 2 3 4]


In [5]:
from cemct.permeability import analyse_permeability


result_x = analyse_permeability(
    volume=volume,
    phase_label=3,
    voxel_size_um=0.7,
    directions=("X",),
    boundary_width=3,
    dynamic_viscosity_pa_s=1.0,
    inlet_pressure_pa=1.0,
    outlet_pressure_pa=0.0,
)

result_x["summary"]

,Direction,Status,Selected phase voxels,Through-connected voxels,Directional connectivity (%),Network pores,Network throats,Flow rate (m^3/s),Pressure drop (Pa),Simulation length (m),Cross-sectional area (m^2),Intrinsic permeability (m^2),Intrinsic permeability (Darcy),Intrinsic permeability (mD)
0,X,Solved,150000,88757,59.171333,80,95,3.793358e-18,1.0,0.000074,4.900000e-09,5.744228e-14,0.058203,58.203386


In [6]:
summary_x = result_x["summary"]

summary_x.T

,0
Direction,X
Status,Solved
Selected phase voxels,150000
Through-connected voxels,88757
Directional connectivity (%),59.171333
Network pores,80
Network throats,95
Flow rate (m^3/s),0.0
Pressure drop (Pa),1.0
Simulation length (m),0.000074


In [7]:
permeability_x = summary_x.loc[
    summary_x["Direction"] == "X",
    "Intrinsic permeability (m^2)",
].iloc[0]

print("X-direction permeability:", permeability_x, "m^2")

X-direction permeability: 5.744227771428786e-14 m^2


In [8]:
result_xyz = analyse_permeability(
    volume=volume,
    phase_label=3,
    voxel_size_um=0.7,
    directions=("X", "Y", "Z"),
    boundary_width=3,
    dynamic_viscosity_pa_s=1.0,
    inlet_pressure_pa=1.0,
    outlet_pressure_pa=0.0,
)

summary_xyz = result_xyz["summary"]

summary_xyz

,Direction,Status,Selected phase voxels,Through-connected voxels,Directional connectivity (%),Network pores,Network throats,Flow rate (m^3/s),Pressure drop (Pa),Simulation length (m),Cross-sectional area (m^2),Intrinsic permeability (m^2),Intrinsic permeability (Darcy),Intrinsic permeability (mD)
0,X,Solved,150000,88757,59.171333,80,95,3.793358e-18,1.0,0.000074,4.900000e-09,5.744228e-14,0.058203,58.203386
1,Y,Solved,150000,88757,59.171333,68,83,4.520909e-18,1.0,0.000074,4.900000e-09,6.845948e-14,0.069367,69.366571
2,Z,Solved,150000,88757,59.171333,82,97,5.157541e-18,1.0,0.000074,4.900000e-09,7.809990e-14,0.079135,79.134723


In [9]:
summary_xyz[
    [
        "Direction",
        "Status",
        "Directional connectivity (%)",
        "Network pores",
        "Network throats",
        "Intrinsic permeability (m^2)",
        "Intrinsic permeability (Darcy)",
        "Intrinsic permeability (mD)",
    ]
]

,Direction,Status,Directional connectivity (%),Network pores,Network throats,Intrinsic permeability (m^2),Intrinsic permeability (Darcy),Intrinsic permeability (mD)
0,X,Solved,59.171333,80,95,5.744228e-14,0.058203,58.203386
1,Y,Solved,59.171333,68,83,6.845948e-14,0.069367,69.366571
2,Z,Solved,59.171333,82,97,7.809990e-14,0.079135,79.134723


In [10]:
print(type(result_x))
print(result_x.keys())

<class 'dict'>
dict_keys(['summary', 'connectivity', 'networks', 'snow_results', 'flow_algorithms', 'pressure_fields', 'metadata'])


In [11]:
from cemct.export import export_analysis_bundle


exported_paths = export_analysis_bundle(
    input_file=file_path,
    analysis_name="permeability_X",
    tables={
        "Permeability Summary": result_x["summary"],
    },
    masks={
        "through_connected_X": (
            result_x["connectivity"]["masks"]["through_connected_X"]
        ),
    },
    arrays={
        "X_pressure": result_x["pressure_fields"]["X"],
    },
    metadata=result_x["metadata"],
)

exported_paths

{'output_directory': WindowsPath('C:/Users/zixia/XseeO2_coding_folder/synthetic_labelled_cement_4phase_100_cemct_results'),
 'binary_masks': {'through_connected_X': WindowsPath('C:/Users/zixia/XseeO2_coding_folder/synthetic_labelled_cement_4phase_100_cemct_results/synthetic_labelled_cement_4phase_100_permeability_X_through_connected_X.tif')},
 'scalar_volumes': {},
 'excel': WindowsPath('C:/Users/zixia/XseeO2_coding_folder/synthetic_labelled_cement_4phase_100_cemct_results/synthetic_labelled_cement_4phase_100_permeability_X_results.xlsx'),
 'metadata_json': WindowsPath('C:/Users/zixia/XseeO2_coding_folder/synthetic_labelled_cement_4phase_100_cemct_results/synthetic_labelled_cement_4phase_100_permeability_X_metadata.json'),
 'array_archive': WindowsPath('C:/Users/zixia/XseeO2_coding_folder/synthetic_labelled_cement_4phase_100_cemct_results/synthetic_labelled_cement_4phase_100_permeability_X_arrays.npz')}

In [12]:
from pathlib import Path


for key, value in exported_paths.items():
    print(key, ":", value)

output_directory : C:\Users\zixia\XseeO2_coding_folder\synthetic_labelled_cement_4phase_100_cemct_results
binary_masks : {'through_connected_X': WindowsPath('C:/Users/zixia/XseeO2_coding_folder/synthetic_labelled_cement_4phase_100_cemct_results/synthetic_labelled_cement_4phase_100_permeability_X_through_connected_X.tif')}
scalar_volumes : {}
excel : C:\Users\zixia\XseeO2_coding_folder\synthetic_labelled_cement_4phase_100_cemct_results\synthetic_labelled_cement_4phase_100_permeability_X_results.xlsx
metadata_json : C:\Users\zixia\XseeO2_coding_folder\synthetic_labelled_cement_4phase_100_cemct_results\synthetic_labelled_cement_4phase_100_permeability_X_metadata.json
array_archive : C:\Users\zixia\XseeO2_coding_folder\synthetic_labelled_cement_4phase_100_cemct_results\synthetic_labelled_cement_4phase_100_permeability_X_arrays.npz


In [13]:
output_directory = exported_paths["output_directory"]

print("Output directory:", output_directory)
print("Directory exists:", output_directory.exists())

for output_file in sorted(output_directory.iterdir()):
    print(output_file.name)

Output directory: C:\Users\zixia\XseeO2_coding_folder\synthetic_labelled_cement_4phase_100_cemct_results
Directory exists: True
synthetic_labelled_cement_4phase_100_permeability_X_arrays.npz
synthetic_labelled_cement_4phase_100_permeability_X_metadata.json
synthetic_labelled_cement_4phase_100_permeability_X_results.xlsx
synthetic_labelled_cement_4phase_100_permeability_X_through_connected_X.tif
